# GuidePath — RT-DETR training and evidence export
ASTLE JOE A S · Run cells in order. Select **Runtime → Change runtime type → T4 GPU**.

The UI demo does not require training. This notebook is for producing the real model required by RAP. One epoch is a smoke experiment, not proof of performance. Checkpoints go to mounted Drive; Colab runtime files can disappear. No 3–5 minute completion guarantee is made.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, zipfile, hashlib
from google.colab import files, drive
drive.mount('/content/drive')
persist = Path('/content/drive/MyDrive/GuidePath')
persist.mkdir(parents=True, exist_ok=True)
work = Path('/content/guidepath_work')
work.mkdir(exist_ok=True)
def extract_checked(archive, destination):
    destination=Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            if not (destination/member.filename).resolve().is_relative_to(destination):
                raise ValueError('Unsafe archive path')
        z.extractall(destination)
if not (work/'guidepath'/'README.md').exists():
    print('Upload GuidePath_GitHub_Ready.zip')
    uploaded=files.upload()
    archives=[n for n in uploaded if n.lower().endswith('.zip')]
    assert len(archives)==1, 'Upload exactly one project ZIP.'
    extract_checked(archives[0],work)
os.chdir(work/'guidepath')
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-dev.txt'],check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print('GPU:',torch.cuda.get_device_name(0))
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)

## Dataset: author archive only
Source: https://github.com/kxzr/WOTR. Review attribution and dataset terms. If Google blocks automation, download from the author's published link in your browser or their alternative, then place `WOTR.zip` in `MyDrive/GuidePath`. Do not silently substitute a dataset.

Keep extracted images on the Colab disk for training speed. Reconnecting to a new runtime requires restoring them. The first download is large.

In [ ]:
import gdown
archive=persist/'WOTR.zip'
if not archive.exists():
    try:
        gdown.download(id='11Idy50HhzedOXxpxYuoecfqMNHGcxVfj',output=str(archive),quiet=False)
    except Exception as exc:
        raise RuntimeError('Download blocked. Obtain WOTR from the author in your browser and upload WOTR.zip to MyDrive/GuidePath; then rerun this cell.') from exc
if not zipfile.is_zipfile(archive):
    raise ValueError('WOTR.zip is missing or incomplete. Replace it with a complete author archive.')
raw=Path('data/raw');raw.mkdir(parents=True,exist_ok=True)
if not list(raw.rglob('Annotations')):extract_checked(archive,raw)
candidates=[p.parent for p in raw.rglob('Annotations') if (p.parent/'JPEGImages').is_dir() and (p.parent/'ImageSets'/'Main'/'train.txt').is_file()]
assert len(candidates)==1, f'Expected one VOC root, found {candidates}. Inspect archive structure.'
source=candidates[0]
print('VOC source:',source)
h=hashlib.sha256()
with archive.open('rb') as f:
    for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
(persist/'archive_sha256.txt').write_text(h.hexdigest()+'  WOTR.zip\n')

## Audit and explicit cleanup
The audit is deliberately blocking. Automatic cleanup can exclude whole images with invalid selected-class boxes and exact pixel duplicates. It retains test, then validation, then train copies. It cannot prove independence of locations or near duplicates. Review those and selected labels before training.

In [ ]:
from scripts.prepare_wotr import prepare
from scripts.clean_dataset import cleanup
original=Path('data/guidepath_original')
if not (original/'audit.json').exists():
    audit=prepare(source,original)
else:audit=json.loads((original/'audit.json').read_text())
print('ORIGINAL AUDIT:',json.dumps(audit,indent=2))
dataset=original
if audit['blocking_issues']:
    cleaned=Path('data/guidepath_clean')
    if not (cleaned/'audit.json').exists():audit=cleanup(original,source,cleaned)
    else:audit=json.loads((cleaned/'audit.json').read_text())
    dataset=cleaned
print('SELECTED AUDIT:',json.dumps(audit,indent=2))
assert not audit['blocking_issues'], 'Inspect remaining issues; do not suppress this check.'
data_yaml=dataset/'data.yaml'

In [ ]:
# Inspect images WITH converted labels, including paving-positive cases.
from PIL import Image,ImageDraw
from IPython.display import display
import yaml
for label in sorted((dataset/'labels/train').glob('*.txt')):
    if not any(line.startswith('0 ') for line in label.read_text().splitlines()):continue
    paths=list((dataset/'images/train').glob(label.stem+'.*'))
    im=Image.open(paths[0]).convert('RGB');draw=ImageDraw.Draw(im);w,h=im.size
    for line in label.read_text().splitlines():
        cls,x,y,bw,bh=map(float,line.split());box=[(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h]
        draw.rectangle(box,outline=['lime','orange','cyan'][int(cls)],width=3)
        draw.text((box[0],box[1]),['tactile_paving','bicycle','motorcycle'][int(cls)],fill='black',stroke_fill='white',stroke_width=1)
    print(label.stem);im.thumbnail((850,600));display(im);break

## Train to Drive or resume
Defaults below are an initial 30-epoch experiment. Set `EPOCHS=1` and `RUN_NAME='smoke'` only for a quick integration check. Batch 8 is optional if memory permits; batch 4 is the conservative starting point. Larger batches are not guaranteed to finish faster.

For recovery set `RESUME` to your Drive run's `weights/last.pt`. Resume restores that run's schedule; changing EPOCHS does not shorten it. Keep outputs from validation decisions separate from the final test.

In [ ]:
EPOCHS=30
BATCH=4
RUN_NAME='baseline'
RESUME=''  # Example: str(persist/'runs/detect/baseline/weights/last.pt')
project=persist/'runs/detect'
cmd=[sys.executable,'-m','scripts.train','--data',str(data_yaml),'--batch',str(BATCH),'--device','0']
if RESUME:
    cmd+=['--resume',RESUME]
    run=Path(RESUME).parent.parent
else:
    cmd+=['--epochs',str(EPOCHS),'--name',RUN_NAME,'--project',str(project)]
    run=project/RUN_NAME
subprocess.run(cmd,check=True)
weights=run/'weights/best.pt'
assert weights.is_file(), 'A trained checkpoint was not saved.'
print('Selected checkpoint:',weights)

## Validation and observed failures
Run detector metrics and export real predictions. Inspect the images, annotate `truth` / `root_cause` in `cases.json`, and document five actual failures. A limited preview is not a representative evaluation. Freeze the checkpoint and thresholds before using the test split.

In [ ]:
subprocess.run([sys.executable,'-m','scripts.evaluate','--weights',str(weights),'--data',str(data_yaml),'--split','val','--name',RUN_NAME+'_val'],check=True)
subprocess.run([sys.executable,'-m','scripts.collect_cases','--weights',str(weights),'--images',str(dataset/'images/val'),'--split','val','--output','runs/cases_val','--limit','100'],check=True)
print('Review runs/cases_val. Re-run without --limit into a NEW directory for full coverage.')

## Final test — only after validation decisions are frozen
Change the confirmation variable after you have completed the actual review. It prevents accidental test-driven tuning.

In [ ]:
VALIDATION_DECISIONS_FROZEN=False
if VALIDATION_DECISIONS_FROZEN:
    subprocess.run([sys.executable,'-m','scripts.evaluate','--weights',str(weights),'--data',str(data_yaml),'--split','test','--name',RUN_NAME+'_test'],check=True)
else:
    print('Test skipped: finish validation review and freeze thresholds first.')

## Real API check and optional live LLM
The next cell uses the actual trained model with a real dataset image. It does not substitute demo data. Enter provider settings privately to exercise the LLM step. No API key is written to the notebook source.

In [ ]:
subprocess.run([sys.executable,'-m','scripts.install_weights','--source',str(weights)],check=True)
os.environ['GUIDEPATH_WEIGHTS']=str(weights)
os.environ['GUIDEPATH_DEVICE']='0'
os.environ['GUIDEPATH_CONFIG']='thresholds.json'
# OPTIONAL: uncomment and enter your own HTTPS provider configuration.
# from getpass import getpass
# os.environ['LLM_API_URL']=input('HTTPS chat-completions URL: ')
# os.environ['LLM_MODEL']=input('Model name: ')
# os.environ['LLM_API_KEY']=getpass('API key: ')
from fastapi.testclient import TestClient
from guidepath.api import app
sample=next((dataset/'images/val').iterdir())
with sample.open('rb') as f:
    response=TestClient(app).post('/reason',data={'question':'Is the tactile path blocked?'},files={'image':(sample.name,f,'application/octet-stream')})
response.raise_for_status()
real=Path('outputs/real');real.mkdir(parents=True,exist_ok=True)
(real/'reason_response.json').write_text(json.dumps(response.json(),indent=2))
print(json.dumps(response.json(),indent=2))

## Export real artifacts now
This ZIP includes actual local evaluation/case outputs, selected best/last checkpoints and run CSV/metadata from Drive. It excludes raw data and additional historical checkpoints. Keep it in Drive, download it, and merge curated evidence into the GitHub project. Add the checkpoint as a release asset rather than ordinary Git history.

In [ ]:
import shutil
from datetime import datetime,timezone
stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
export=persist/f'GuidePath_real_artifacts_{stamp}.zip'
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as z:
    for base in [Path('runs'),Path('outputs/real'),Path('weights')]:
        if base.exists():
            for p in base.rglob('*'):
                if p.is_file():z.write(p,str(p))
    for p in run.rglob('*'):
        if p.is_file() and (p.suffix!='.pt' or p.name in ('best.pt','last.pt')):
            z.write(p,'training_run/'+str(p.relative_to(run)))
    for name in ['audit.json','manifest.json','cleanup_report.json','data.yaml']:
        p=dataset/name
        if p.exists():z.write(p,'dataset/'+name)
    z.write('thresholds.json','thresholds.json')
print('Preserved:',export)
files.download(str(export))